# Oracle Delta Sharing CDF Smoke Test

This notebook reads **Change Data Feed (CDF)** from an Oracle Autonomous Database
that publishes a Delta Sharing endpoint, and displays the raw change rows.

## Oracle's file-level CDF

Oracle implements CDF at the **file level**, not the row level. When *any* row in a
data file changes, Oracle replaces the **entire file**. The CDF response therefore
shows:
- **All rows from the old file** as `delete`
- **All rows from the new file** as `insert`

Unchanged rows appear as matching DELETE + INSERT pairs (file-level artifacts).
To derive the true net changes, cancel out identical pairs — this notebook shows
the raw output so you can observe this behavior directly.

In [0]:
%pip install delta-sharing -q

In [0]:
# ---------------------------------------------------------------------------
# Oracle Delta Sharing CDF Smoke Test
# ---------------------------------------------------------------------------
# Reads raw CDF (Change Data Feed) for a single version window from an
# Oracle Autonomous Database Delta Sharing endpoint and prints the rows.
#
# This script does NOT write anything to a Delta table — it is purely
# diagnostic, useful for inspecting what Oracle's file-level CDF returns
# before applying changes downstream.
# ---------------------------------------------------------------------------

import json
import time

import pandas as pd
import delta_sharing
from delta_sharing.protocol import DeltaSharingProfile, Table
from delta_sharing.reader import CdfOptions, DeltaSharingReader, to_converters
from delta_sharing.rest_client import DataSharingRestClient
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Configuration  —  replace placeholders with your own values
# ---------------------------------------------------------------------------

# Path to the .share profile file on your Databricks workspace.
# The profile contains the Oracle sharing endpoint URL and credentials.
# See: https://github.com/delta-io/delta-sharing/blob/main/PROTOCOL.md#profile-file-format
PROFILE_PATH = "/Workspace/Users/<your-email>/<your-profile>.share"

# Coordinates of the shared table (as listed by delta_sharing.SharingClient).
SHARE_NAME  = "<YOUR_SHARE>"
SCHEMA_NAME = "<YOUR_SCHEMA>"
TABLE_NAME  = "<YOUR_TABLE>"

# CDF version window to inspect.
# Set both to the same value to see a single version's changes.
START_VERSION = 1
END_VERSION   = 2

# How many sample rows to print.
SHOW_SAMPLE_ROWS = 50

# ---------------------------------------------------------------------------
# Read CDF via Delta Sharing REST API
# ---------------------------------------------------------------------------
# We bypass the high-level library functions (load_table_changes_as_pandas,
# spark.read.format("deltaSharing")) because:
#   1. Oracle omits _commit_timestamp → KeyError in the library
#   2. Spark's Java connector throws InvocationTargetException on serverless
#   3. Pre-signed HTTPS URLs can't be read by spark.read.parquet()
#
# Instead we:
#   a) Call list_table_changes() on the REST client to get pre-signed URLs
#   b) Download each Parquet file via HTTP with _to_pandas()
#   c) Concatenate into a single pandas DataFrame, then convert to Spark
# ---------------------------------------------------------------------------

table_url = f"{PROFILE_PATH}#{SHARE_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
print(f"[info] table_url={table_url}")
print(f"[info] requested CDF window={START_VERSION} -> {END_VERSION}")

t0 = time.time()

profile   = DeltaSharingProfile.read_from_file(PROFILE_PATH)
rest      = DataSharingRestClient(profile)
table_obj = Table(name=TABLE_NAME, share=SHARE_NAME, schema=SCHEMA_NAME)

cdf_opts  = CdfOptions(starting_version=START_VERSION, ending_version=END_VERSION)
response  = rest.list_table_changes(table_obj, cdf_opts)

# Parse the schema returned by Oracle so _to_pandas() can cast columns.
schema_json = json.loads(response.metadata.schema_string)
converters  = to_converters(schema_json)

# Download every Parquet action file returned in the CDF response.
# Each action corresponds to one physical file on Oracle object storage.
pdfs = []
for action in response.actions:
    pdfs.append(
        DeltaSharingReader._to_pandas(
            action,
            converters,
            True,   # for_cdf — adds _change_type / _commit_version columns
            None,   # limit
            True,   # use_delta_format
        )
    )

elapsed = time.time() - t0

# ---------------------------------------------------------------------------
# Display results
# ---------------------------------------------------------------------------

if not pdfs:
    print(f"[info] elapsed_seconds={elapsed:.2f}")
    print("[result] empty CDF response for requested version window")
else:
    cdf_pdf = pd.concat(pdfs, ignore_index=True)
    df = spark.createDataFrame(cdf_pdf)

    # Safety filter: ensure we only look at the requested version range.
    if "_commit_version" in df.columns:
        df = df.filter(
            (F.col("_commit_version") >= F.lit(START_VERSION))
            & (F.col("_commit_version") <= F.lit(END_VERSION))
        )

    raw_count = df.count()
    elapsed = time.time() - t0

    print(f"[info] elapsed_seconds={elapsed:.2f}")
    print(f"[info] raw_row_count={raw_count:,}")
    print(f"[info] columns={df.columns}")

    if raw_count == 0:
        print("[result] empty CDF response for requested version window")
    else:
        # --- Summary: row counts by version and change type ----------------
        if "_commit_version" in df.columns and "_change_type" in df.columns:
            print("[result] rows by commit version and change type")
            (
                df.groupBy("_commit_version", "_change_type")
                  .count()
                  .orderBy("_commit_version", "_change_type")
                  .show(100, truncate=False)
            )

        # --- Sample rows (truncated for readability) -----------------------
        # NOTE: Because Oracle uses file-level CDF, you will typically see
        #   more rows than the actual number of changed rows.  For example,
        #   inserting 1 row into a file that already has 2 rows produces:
        #     2 deletes  (old file: all existing rows)
        #     3 inserts  (new file: existing rows + the new row)
        #   The 2 matching DELETE+INSERT pairs are unchanged — only the
        #   extra INSERT is the real change.
        print("[result] sample rows")
        order_cols = []
        if "_commit_version" in df.columns:
            order_cols.append(F.col("_commit_version").desc())
        if "_change_type" in df.columns:
            order_cols.append(F.col("_change_type"))

        if order_cols:
            df.orderBy(*order_cols).show(SHOW_SAMPLE_ROWS, truncate=True)
        else:
            df.show(SHOW_SAMPLE_ROWS, truncate=True)